# ⚡ Benchmark de Performance da IA

Este notebook avalia a eficiência computacional do motor Minimax + Alpha-Beta, comparando a velocidade de busca por profundidade e testando a eficácia da IA contra um agente que faz jogadas aleatórias.

In [ ]:
import sys
import os
import time
import random
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'backend'))

from games.registry import get_game_class, GAMES
from engine.minimax_ai import MinimaxAI
print("Módulos importados com sucesso!")

## 1. Latência de Busca por Profundidade

Vamos comparar o tempo necessário para calcular a melhor jogada inicial em diferentes profundidades para o Jogo da Velha (Tic-Tac-Toe) e Tapatan.

In [ ]:
def benchmark_depths(game_id, depths):
    game = get_game_class(game_id)
    state = game.get_initial_state()
    
    print(f"=== Benchmark: {game.name} ===")
    print(f"{'Profundidade':<12} | {'Tempo (s)':<12} | {'Jogada Escolhida':<15}")
    print("-" * 46)
    
    for d in depths:
        ai = MinimaxAI(game, max_depth=d, randomize=False)
        t0 = time.perf_counter()
        move = ai.get_best_move(state)
        dt = time.perf_counter() - t0
        print(f"{d:<12} | {dt:<12.5f} | {str(move):<15}")
    print()

benchmark_depths("tictactoe", [1, 3, 5, 7, 9])
benchmark_depths("tapatan", [1, 3, 5])

## 2. Simulação de Partida: Minimax vs Aleatório

Para provar que a IA é eficaz, vamos simular 50 partidas onde o Player 1 (X) joga de forma totalmente aleatória e o Player 2 (O) usa a inteligência Minimax (profundidade 3).

In [ ]:
def play_game(game_id, p1_is_ai, p2_is_ai, ai_depth=3):
    game = get_game_class(game_id)
    state = game.get_initial_state()
    
    ai_player = MinimaxAI(game, max_depth=ai_depth, randomize=True)
    
    while not game.check_result(state)['over']:
        curr = state["current_player"]
        is_ai = p1_is_ai if curr == 1 else p2_is_ai
        
        if is_ai:
            move = ai_player.get_best_move(state)
        else:
            moves = game.get_valid_moves(state)
            move = random.choice(moves) if moves else None
            
        if move is None:
            break
        state = game.apply_move(game.clone_state(state), move)
        
    return game.check_result(state)["winner"]

def run_simulation(game_id, num_rounds=50):
    print(f"Rodando {num_rounds} simulações para {game_id} (Aleatório vs IA)... ")
    results = {1: 0, 2: 0, 0: 0} # 1: Aleatório, 2: IA, 0: Empate
    
    for _ in range(num_rounds):
        winner = play_game(game_id, p1_is_ai=False, p2_is_ai=True, ai_depth=3)
        if winner is not None:
            results[winner] += 1
            
    total = sum(results.values())
    print(f"  Vitórias do Jogador Aleatório (X): {results[1]} ({(results[1]/total)*100:.1f}%)")
    print(f"  Vitórias da IA Minimax (O): {results[2]} ({(results[2]/total)*100:.1f}%)")
    print(f"  Empates: {results[0]} ({(results[0]/total)*100:.1f}%)")
    print()

run_simulation("tictactoe")
run_simulation("tapatan")